### Data Ingestion


In [5]:
###Document Structure
from langchain_core.documents import Document

In [6]:
doc=Document(
    page_content="This is the main text content i am using to create RAG.",
    metadata={"source": "my_file.txt",
              "author": "Somanath Mishra",
              "created_at": "2026-05-09",
              "page": 1}
)
doc

Document(metadata={'source': 'my_file.txt', 'author': 'Somanath Mishra', 'created_at': '2026-05-09', 'page': 1}, page_content='This is the main text content i am using to create RAG.')

In [7]:
## create simple text file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [6]:
simple_texts={
    "../data/text_files/python_intro.txt":""" Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code compared to other programming languages.
    key Features:
    - Easy to learn and use
    - Versatile and powerful
    - Large standard library
    - Strong community support.
    """,
    "../data/text_files/machine_learning.txt":""" Machine learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It involves training a model on a dataset, allowing it to learn patterns and make predictions or decisions based on new data. Machine learning is widely used in various applications, including image recognition, natural language processing, and recommendation systems.
    key Concepts:
    - Supervised Learning
    - Unsupervised Learning
    - Reinforcement Learning
    """
}

for file_path, content in simple_texts.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
print("sample text files created successfully.")

sample text files created successfully.


In [8]:
import torch
print(torch.__version__)

2.2.0+cpu


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("ok")

ok


In [10]:
### Text Loader
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content=" Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code compared to other programming languages.\n    key Features:\n    - Easy to learn and use\n    - Versatile and powerful\n    - Large standard library\n    - Strong community support.\n    ")]


In [10]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all text files from the directory
dir_loader = DirectoryLoader(
    "../data/text_files",
     glob="**/*.txt",
      loader_cls=TextLoader,
      loader_kwargs={"encoding": "utf-8"},
      show_progress=False
      )

documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content=' Machine learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It involves training a model on a dataset, allowing it to learn patterns and make predictions or decisions based on new data. Machine learning is widely used in various applications, including image recognition, natural language processing, and recommendation systems.\n    key Concepts:\n    - Supervised Learning\n    - Unsupervised Learning\n    - Reinforcement Learning\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content=" Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability and a syntax that allows 

In [11]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

In [12]:
from langchain_community.document_loaders import PyMuPDFLoader

##load pdf using PyMuPDFLoader
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Pol

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [15]:
chunks=split_documents(pdf_documents)
chunks

Split 38 documents into 155 chunks

Example chunk:
Content: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz...
Metadata: {'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}


[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Pol

### embeddings and vectorStoreDB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document

In [31]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully: {self.model.get_sentence_embedding_dimension()} dimensions")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, documents) -> np.ndarray:
        """Generate embeddings for documents or plain strings."""
        if not self.model:
            raise ValueError("Model not loaded. Call load_model() first.")
        
        # handle both strings and Document objects
        if isinstance(documents[0], str):
            texts = documents
        else:
            texts = [doc.page_content for doc in documents]
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully with shape: {embeddings.shape}")
        return embeddings


# Initialize
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")

Loading model: all-MiniLM-L6-v2
Model loaded successfully: 384 dimensions


VectorStore

In [32]:
import chromadb
print(chromadb.__version__)

0.4.24


In [33]:
import os
import uuid
import numpy as np
import chromadb
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [34]:
class VectorStore:
    """Manage document embeddings in a ChromaDB."""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory  # ← was missing, caused AttributeError
        self.client = None
        self.collection = None
        self.__initialize_store()

    def __initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for rag"}  # ← removed extra `"}`
            )
            print(f"Vector store initialized successfully with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store."""
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        documents_texts = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  # ← space → underscore in ID
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)  # ← renamed from content_page
            metadatas.append(metadata)

            # Document content
            documents_texts.append(doc.page_content)

            # Embedding
            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embedding_list
            )
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vector_store = VectorStore()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store initialized successfully with collection: pdf_documents
Existing documents in collection: 155


In [35]:
###Extract all the text from chunks and create generate embbedings
texts=[doc.page_content for doc in chunks]
texts

['Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superior in quality while being more parallelizable and requiring signiﬁcantl

In [15]:
chunks

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Pol

In [23]:
# pass chunks directly, not texts
embeddings = embedding_manager.generate_embeddings(chunks)

# store in vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 155 texts...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings generated successfully with shape: (155, 384)
Adding 155 documents to the vector store...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Documents added successfully. Total documents in collection: 155


Retriever Pipeline From VectorStore

In [43]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - (distance / 2)

                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embedding_manager)



In [44]:
rag_retriever

In [46]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_24d40593_25',
  'content': 'convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5\nTraining\nThis section describes the training regime for our models.\n5.1\nTraining Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [3], which has a shared source-\ntarget vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT\n2014 English-French dataset consisting of 36M sentences and split tokens

### RAG Pipeline- VectorDB To LLM Output Generation